# Hackathon Data Access

This notebook gets you set up with the hackathon datasets. It handles:
- Installing dependencies
- Configuring credentials
- Browsing available datasets
- Downloading data with resume support and checksum verification

Works on: **Colab**, **Kaggle**, **RunPod**, and **local machines**.

## 1. Install dependencies

In [ ]:
!pip install -q boto3 tqdm

## 2. Get the helper module

If you're on Colab/Kaggle and don't have `r2_download.py` locally, this cell downloads it.

In [9]:
import os
from pathlib import Path

HELPER_URL = "https://raw.githubusercontent.com/SALA-AI-LATAM/hackathon-participants/main/r2_download.py"

if not Path("r2_download.py").exists():
    print("Downloading r2_download.py...")
    import urllib.request
    urllib.request.urlretrieve(HELPER_URL, "r2_download.py")
    print("Done.")
else:
    print("r2_download.py already present.")

r2_download.py already present.


## 3. Configure credentials

Organizers will provide a `participant-download.env` file with R2 credentials.

**Option A (recommended):** Upload the file to Colab (drag it into the file panel on the left), then run the next cell.

**Option B:** Edit the placeholder values in the next cell directly.

In [10]:
# === Set R2 credentials ===
# Option A: Upload the participant-download.env file provided by organizers
#           (drag it into the Colab file panel, then run this cell)
ENV_FILE = "participant-download.env"
if Path(ENV_FILE).exists():
    for line in open(ENV_FILE):
        line = line.strip()
        if line and not line.startswith("#"):
            key, val = line.removeprefix("export ").split("=", 1)
            os.environ[key] = val.strip('"')
    print(f"Credentials loaded from {ENV_FILE}")
else:
    # Option B: Paste credentials directly (organizers will provide these)
    os.environ["R2_ENDPOINT"] = "https://YOUR_ACCOUNT_ID.r2.cloudflarestorage.com"
    os.environ["R2_ACCESS_KEY_ID"] = "YOUR_ACCESS_KEY"
    os.environ["R2_SECRET_ACCESS_KEY"] = "YOUR_SECRET_KEY"
    os.environ["R2_BUCKET"] = "sala-2026-hackathon-data"
    print("Using inline credentials (edit the values above if they say YOUR_...)")

Credentials loaded from participant-download.env


## 4. Import helper and create client

In [11]:
import r2_download as hd

print(f"Detected environment: {hd._detect_environment()}")
print(f"Default data directory: {hd._default_data_dir()}")

client = hd.get_s3_client()

Detected environment: local
Default data directory: ./hackathon_data


## 5. Load manifest and browse datasets

In [12]:
manifest = hd.load_manifest(
    bucket=os.environ["R2_BUCKET"],
    s3_client=client,
    cache_path="manifest.json",  # caches locally to avoid re-downloading
)

hd.summarize_manifest(manifest)

Dataset               Shards  Size (GB) Format     Description
--------------------------------------------------------------------------------
precipitation-nowcasting      14       2.38 csv+netcdf+pt Weather station CSVs + LDAS gr
marine-acoustic-colab       1       0.44 tar.gz     Colab-friendly subset of under
marine-acoustic-core     146       7.33 wav        Curated subset of underwater a
bruv-videos               18      69.48 mp4        BRUV underwater video files — 


## 6. List shards for a specific dataset

In [20]:
# Pick a dataset to explore — change this to your track
# Available: "precipitation-nowcasting" (~2.4 GB), "marine-acoustic-core" (~7.3 GB), "bruv-videos" (~65 GB)
DATASET = "marine-acoustic-core"  # start small

shards = hd.list_shards(manifest, dataset=DATASET)
print(f"Dataset: {DATASET}")
print(f"Shards: {len(shards)}")
print(f"Total size: {sum(s['size_bytes'] for s in shards) / 1e9:.2f} GB")
print()

# Show first few shards
for s in shards[:5]:
    print(f"  {s['key']}  ({s['size_bytes'] / 1e6:.1f} MB)  tags={s.get('tags', [])}")
if len(shards) > 5:
    print(f"  ... and {len(shards) - 5} more")

Dataset: marine-acoustic-core
Shards: 146
Total size: 7.33 GB

  marine-acoustic/5783/5783.040122170635.log.xml  (0.0 MB)  tags=['core', '5783', 'metadata']
  marine-acoustic/5783/5783.040122170635.wav  (345.6 MB)  tags=['core', '5783']
  marine-acoustic/5783/5783.040122220635.log.xml  (0.0 MB)  tags=['core', '5783', 'metadata']
  marine-acoustic/5783/5783.040122220635.wav  (345.6 MB)  tags=['core', '5783']
  marine-acoustic/5783/5783.040123030635.log.xml  (0.0 MB)  tags=['core', '5783', 'metadata']
  ... and 141 more


## 7. Download a dataset

Downloads with progress bars, resume support, and SHA-256 verification.
If interrupted, re-run the cell — it picks up where it left off.

In [21]:
stats = hd.download_dataset(
    manifest,
    dataset_name=DATASET,
    # dest_dir="./my_data",  # uncomment to override default
    # tags=["train"],         # uncomment to download only specific tags
)

print(f"\nDownloaded: {stats['downloaded']}, Skipped: {stats['skipped']}, Failed: {stats['failed']}")
if stats["errors"]:
    print("Errors:")
    for e in stats["errors"]:
        print(f"  {e['key']}: {e['error']}")

Shards:   0%|          | 0/146 [00:00<?, ?shard/s, file=5783.040122170635.log.xml, size_mb=0]
5783.040122170635.log.xml:   0%|          | 0.00/2.36k [00:00<?, ?B/s]
5783.040122170635.log.xml: 100%|██████████| 2.36k/2.36k [00:00<00:00, 3.29kB/s]
Shards:   1%|          | 1/146 [00:01<02:41,  1.12s/shard, file=5783.040122170635.wav, size_mb=346]  
5783.040122170635.wav:   0%|          | 0.00/346M [00:00<?, ?B/s]
5783.040122170635.wav:   0%|          | 262k/346M [00:00<08:51, 650kB/s]
5783.040122170635.wav:   1%|          | 2.88M/346M [00:00<00:46, 7.31MB/s]
5783.040122170635.wav:   2%|▏         | 6.03M/346M [00:00<00:24, 13.8MB/s]
5783.040122170635.wav:   3%|▎         | 9.70M/346M [00:00<00:17, 19.8MB/s]
5783.040122170635.wav:   4%|▍         | 13.4M/346M [00:00<00:13, 24.0MB/s]
5783.040122170635.wav:   5%|▌         | 17.3M/346M [00:00<00:11, 27.7MB/s]
5783.040122170635.wav:   6%|▌         | 21.5M/346M [00:01<00:10, 30.3MB/s]
5783.040122170635.wav:   8%|▊         | 26.2M/346M [00:01<00:09,


Downloaded: 146, Skipped: 0, Failed: 0


## 8. Verify downloads

In [22]:
# Re-run download with verify=True to check all existing files
# (skips downloading, just verifies checksums)
verify_stats = hd.download_dataset(
    manifest,
    dataset_name=DATASET,
    resume=True,
    verify=True,
)
print(f"All files verified: {verify_stats['failed'] == 0}")

Shards: 100%|██████████| 146/146 [00:00<00:00, 2500.84shard/s, file=201010_5942.wav, size_mb=29]

All files verified: True


## 9. Inspect the data

Load one shard and peek at the structure. Adjust based on the data format.

In [24]:
from pathlib import Path

data_dir = Path(hd._default_data_dir())
first_shard = shards[1]["key"]
shard_path = data_dir / first_shard

print(f"Shard: {shard_path}")
print(f"Size: {shard_path.stat().st_size / 1e6:.1f} MB")

# === Uncomment the loader that matches your data format ===

# Parquet:
# import pandas as pd
# df = pd.read_parquet(shard_path)
# print(f"Shape: {df.shape}")
# df.head()

# CSV:
# import pandas as pd
# df = pd.read_csv(shard_path)
# print(f"Shape: {df.shape}")
# df.head()

# tar.gz (list contents):
# import tarfile
# with tarfile.open(shard_path, 'r:gz') as tar:
#     for member in tar.getmembers()[:10]:
#         print(f"  {member.name}  ({member.size} bytes)")

Shard: hackathon_data/marine-acoustic/5783/5783.040122170635.wav
Size: 345.6 MB


## 10. Next steps

You now have the data downloaded and verified. From here:

1. **Explore** the data — look at distributions, missing values, samples
2. **Build your pipeline** — data loading, preprocessing, feature engineering
3. **Train models** — start simple, iterate

Tips:
- If your download was interrupted, just re-run cell 7 — it resumes automatically
- Use `tags` parameter to download only the subset you need
- On Colab, data persists for the session. Re-download if you reconnect.
- On RunPod, data in `/workspace` persists on the network volume across sessions.